In [2]:
import numpy as np
import cv2
import torch
import torchvision
from PIL import Image
import os
import copy
import torch
import torchvision
import torch.nn as nn
import scipy
import torchvision.transforms as transforms
from torchvision import datasets as ds
from torch.utils.data import DataLoader,Dataset,Subset,ConcatDataset,random_split
import matplotlib.pyplot as plt
import numpy as np
import random
import pandas as pd
from PIL import Image
import cv2
import matplotlib.pyplot as plt
import tqdm
import celeba_dataset

In [3]:
print(os.getcwd())

/home/zhixin/Project/FL_CelebA_10_27


In [4]:
# 把数据缩放到（-1，1）
class Oneone(torch.nn.Module):
    def __init__(self, inplace=False):
        super().__init__()
        self.inplace = inplace

    def forward(self, tensor):
        return tensor*2.0-1.0
        # return F.normalize(tensor, self.mean, self.std, self.inplace)

# transform = transforms.Compose是把一系列图片操作组合起来，比如减去像素均值等。
# DataLoader读入的数据类型是PIL.Image
# 这里对图片不做任何处理，仅仅是把PIL.Image转换为torch.FloatTensor，从而可以被pytorch计算
transform_train = transforms.Compose(
    [
        transforms.CenterCrop((150,200)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
        Oneone(),
    ]
)
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    Oneone(),
])

In [5]:
load_model = True
load_data_loader = False
learning_rate = 0.005
batch_size = 128
trigger_size = 8
trigger_pos = 0
inject_r = 0.1
untrust_prop = 0.95
ret = 175# ret是控制mask透明度的阈值（175）
target_label_1 = 9
target_label_2 = 0

In [22]:
cfg = {
    'VGG11': [64, 'M', 128, 'M', 256, 256, 'M', 512, 512, 'M', 512, 512, 'M'],
    'VGG13': [64, 64, 'M', 128, 128, 'M', 256, 256, 'M', 512, 512, 'M', 512, 512, 'M'],
    'VGG16': [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 'M', 512, 512, 512, 'M', 512, 512, 512, 'M'],
    'VGG19': [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 256, 'M', 512, 512, 512, 512, 'M', 512, 512, 512, 512, 'M'],
}

intermediate_result = {}
net_name = "VGG19"

class VGG(nn.Module):
    def __init__(self, vgg_name):
        super(VGG, self).__init__()
        self.features = self._make_layers(cfg[vgg_name])
        self.classifier = nn.Linear(12288, 2)
        global intermediate_result

    def forward(self, x):
        seq = self.features
        out = x
        for i,layer in enumerate(seq):
            out = layer(out)

            if type(layer) == torch.nn.modules.conv.Conv2d:
                intermediate_result[str(i)] = out
#         out = self.features(x)
        out = out.view(out.size(0), -1)
        intermediate_result["linear"] = out
        out = self.classifier(out)
        return out

    def _make_layers(self, cfg):
        layers = []
        in_channels = 3
        for x in cfg:
            if x == 'M':
                layers += [nn.MaxPool2d(kernel_size=2, stride=2)]
            else:
                layers += [nn.Conv2d(in_channels, x, kernel_size=3, padding=1),
                           nn.BatchNorm2d(x),
                           nn.ReLU(inplace=True)]
                in_channels = x
        layers += [nn.AvgPool2d(kernel_size=1, stride=1)]
        return nn.Sequential(*layers)


In [ ]:
class Vgg_face_dag(nn.Module):

    def __init__(self):
        super(Vgg_face_dag, self).__init__()
        self.meta = {'mean': [129.186279296875, 104.76238250732422, 93.59396362304688],
                     'std': [1, 1, 1],
                     'imageSize': [224, 224, 3]}
        self.conv1_1 = nn.Conv2d(3, 64, kernel_size=[3, 3], stride=(1, 1), padding=(1, 1))
        self.relu1_1 = nn.ReLU(inplace=True)
        self.conv1_2 = nn.Conv2d(64, 64, kernel_size=[3, 3], stride=(1, 1), padding=(1, 1))
        self.relu1_2 = nn.ReLU(inplace=True)
        self.pool1 = nn.MaxPool2d(kernel_size=[2, 2], stride=[2, 2], padding=0, dilation=1, ceil_mode=False)
        self.conv2_1 = nn.Conv2d(64, 128, kernel_size=[3, 3], stride=(1, 1), padding=(1, 1))
        self.relu2_1 = nn.ReLU(inplace=True)
        self.conv2_2 = nn.Conv2d(128, 128, kernel_size=[3, 3], stride=(1, 1), padding=(1, 1))
        self.relu2_2 = nn.ReLU(inplace=True)
        self.pool2 = nn.MaxPool2d(kernel_size=[2, 2], stride=[2, 2], padding=0, dilation=1, ceil_mode=False)
        self.conv3_1 = nn.Conv2d(128, 256, kernel_size=[3, 3], stride=(1, 1), padding=(1, 1))
        self.relu3_1 = nn.ReLU(inplace=True)
        self.conv3_2 = nn.Conv2d(256, 256, kernel_size=[3, 3], stride=(1, 1), padding=(1, 1))
        self.relu3_2 = nn.ReLU(inplace=True)
        self.conv3_3 = nn.Conv2d(256, 256, kernel_size=[3, 3], stride=(1, 1), padding=(1, 1))
        self.relu3_3 = nn.ReLU(inplace=True)
        self.pool3 = nn.MaxPool2d(kernel_size=[2, 2], stride=[2, 2], padding=0, dilation=1, ceil_mode=False)
        self.conv4_1 = nn.Conv2d(256, 512, kernel_size=[3, 3], stride=(1, 1), padding=(1, 1))
        self.relu4_1 = nn.ReLU(inplace=True)
        self.conv4_2 = nn.Conv2d(512, 512, kernel_size=[3, 3], stride=(1, 1), padding=(1, 1))
        self.relu4_2 = nn.ReLU(inplace=True)
        self.conv4_3 = nn.Conv2d(512, 512, kernel_size=[3, 3], stride=(1, 1), padding=(1, 1))
        self.relu4_3 = nn.ReLU(inplace=True)
        self.pool4 = nn.MaxPool2d(kernel_size=[2, 2], stride=[2, 2], padding=0, dilation=1, ceil_mode=False)
        self.conv5_1 = nn.Conv2d(512, 512, kernel_size=[3, 3], stride=(1, 1), padding=(1, 1))
        self.relu5_1 = nn.ReLU(inplace=True)
        self.conv5_2 = nn.Conv2d(512, 512, kernel_size=[3, 3], stride=(1, 1), padding=(1, 1))
        self.relu5_2 = nn.ReLU(inplace=True)
        self.conv5_3 = nn.Conv2d(512, 512, kernel_size=[3, 3], stride=(1, 1), padding=(1, 1))
        self.relu5_3 = nn.ReLU(inplace=True)
        self.pool5 = nn.MaxPool2d(kernel_size=[2, 2], stride=[2, 2], padding=0, dilation=1, ceil_mode=False)
        self.fc6 = nn.Linear(in_features=25088, out_features=4096, bias=True)
        self.relu6 = nn.ReLU(inplace=True)
        self.dropout6 = nn.Dropout(p=0.5)
        self.fc7 = nn.Linear(in_features=4096, out_features=4096, bias=True)
        self.relu7 = nn.ReLU(inplace=True)
        self.dropout7 = nn.Dropout(p=0.5)
        self.fc8 = nn.Linear(in_features=4096, out_features=2622, bias=True)

    def forward_once(self, x0):
        x1 = self.conv1_1(x0)
        x2 = self.relu1_1(x1)
        x3 = self.conv1_2(x2)
        x4 = self.relu1_2(x3)
        x5 = self.pool1(x4)
        x6 = self.conv2_1(x5)
        x7 = self.relu2_1(x6)
        x8 = self.conv2_2(x7)
        x9 = self.relu2_2(x8)
        x10 = self.pool2(x9)
        x11 = self.conv3_1(x10)
        x12 = self.relu3_1(x11)
        x13 = self.conv3_2(x12)
        x14 = self.relu3_2(x13)
        x15 = self.conv3_3(x14)
        x16 = self.relu3_3(x15)
        x17 = self.pool3(x16)
        x18 = self.conv4_1(x17)
        x19 = self.relu4_1(x18)
        x20 = self.conv4_2(x19)
        x21 = self.relu4_2(x20)
        x22 = self.conv4_3(x21)
        x23 = self.relu4_3(x22)
        x24 = self.pool4(x23)
        x25 = self.conv5_1(x24)
        x26 = self.relu5_1(x25)
        x27 = self.conv5_2(x26)
        x28 = self.relu5_2(x27)
        x29 = self.conv5_3(x28)
        x30 = self.relu5_3(x29)
        x31_preflatten = self.pool5(x30)
        x31 = x31_preflatten.view(x31_preflatten.size(0), -1)
        x32 = self.fc6(x31)
        x33 = self.relu6(x32)
        x34 = self.dropout6(x33)
        x35 = self.fc7(x34)
        x36 = self.relu7(x35)
        x37 = self.dropout7(x36)
        x38 = self.fc8(x37)
        return x38

    def forward(self,input1,input2):
        output1 = self.forward_once(input1)
        output2 = self.forward_once(input2)

        #euclidean_distance = F.pairwise_distance(output1, output2, keepdim = True)
        difference = output1 - output2
        return difference

def vgg_face_dag(weights_path=None, **kwargs):
    """
    load imported model instance

    Args:
        weights_path (str): If set, loads model weights from the given path
    """
    model = Vgg_face_dag()
    if weights_path:
        state_dict = torch.load(weights_path)
        model.load_state_dict(state_dict)
    return model

In [23]:
net = VGG(net_name)
print(net)
# 定义损失函数和优化器
criterion = torch.nn.CrossEntropyLoss()
# optimizer = torch.optim.Adam(net.parameters(), lr=learning_rate)
optimizer = torch.optim.SGD(net.parameters(), lr=learning_rate, momentum=0.9, weight_decay=5e-4)
optimizer_2 = torch.optim.SGD(net.parameters(), lr=learning_rate, momentum=0.9, weight_decay=5e-4)
# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200)



# 如果有gpu就使用gpu，否则使用cpu
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
net = net.to(device)

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (9): ReLU(inplace=True)
    (10): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (12): ReLU(inplace=True)
    (13): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (14): Conv2d(128, 256

#  加载CelebA数据集

In [8]:
#加载原始数据集
celebA_train_dataset=celeba_dataset.CelebA(root='../', split='train',target_type='attr', transform=transform_train, target_transform=None, download=True)
celebA_val_dataset=celeba_dataset.CelebA(root='../', split='valid',target_type='attr',transform=transform_train, target_transform=None, download=True)

#取部分数据集
celebA_train_dataset=random_split(celebA_train_dataset,
                                  lengths=[int(len(celebA_train_dataset)*0.25),len(celebA_train_dataset)-int(len(celebA_train_dataset)*0.25)])[0]

celebA_train_loader=DataLoader(dataset = celebA_train_dataset,
                              batch_size=batch_size,
                              shuffle=True)
celebA_val_loader=DataLoader(dataset = celebA_val_dataset,
                              batch_size=batch_size,
                              shuffle=True)

Files already downloaded and verified
Files already downloaded and verified


In [9]:
print("标签参数如下")
for i,name in enumerate(celebA_val_dataset.attr_names):
    print(str(i)+" "+name)

print("训练集长度："+str(len(celebA_train_dataset)) )
print("验证集长度："+str(len(celebA_val_dataset)) )

标签参数如下
0 5_o_Clock_Shadow
1 Arched_Eyebrows
2 Attractive
3 Bags_Under_Eyes
4 Bald
5 Bangs
6 Big_Lips
7 Big_Nose
8 Black_Hair
9 Blond_Hair
10 Blurry
11 Brown_Hair
12 Bushy_Eyebrows
13 Chubby
14 Double_Chin
15 Eyeglasses
16 Goatee
17 Gray_Hair
18 Heavy_Makeup
19 High_Cheekbones
20 Male
21 Mouth_Slightly_Open
22 Mustache
23 Narrow_Eyes
24 No_Beard
25 Oval_Face
26 Pale_Skin
27 Pointy_Nose
28 Receding_Hairline
29 Rosy_Cheeks
30 Sideburns
31 Smiling
32 Straight_Hair
33 Wavy_Hair
34 Wearing_Earrings
35 Wearing_Hat
36 Wearing_Lipstick
37 Wearing_Necklace
38 Wearing_Necktie
39 Young
40 
训练集长度：40692
验证集长度：19867


In [10]:
#数据切分加工
def get_split_indices(dataset,target_index):
    '''
    按标签划分数据集，返回数据集的下标集合
    :param dataset:
    :param target_index: 按照这个下标的标签进行划分
    :return: 返回的list中，每个元素代表不同值的标签下标的集合
    '''
    group1=[]
    group2=[]
    for i,(data,target) in enumerate(dataset):
        if target[target_index]==0:
            group1.append(i)
        else:
            group2.append(i)
    return [group1,group2]
def shuffle_dataset(dataset,target_index):
    '''
    打乱数据集，把目标标签取反
    :param dataset:
    :param target_index: 标签下标
    :return:
    '''
    dataset=copy.deepcopy(dataset)
    for i,(data,target) in enumerate(dataset):
        dataset[i][1][target_index]=target[target_index]^1 #取反操作
    return dataset

In [11]:
#标签的下标
target_index=31
#身份标签的下标
group_index=20
#shuffle数据集的长度
shuffle_len=200

In [12]:
indices=get_split_indices(celebA_train_dataset,target_index)
celebA_train_target1_dataset=Subset(celebA_train_dataset,indices[0])
celebA_train_target2_dataset=Subset(celebA_train_dataset,indices[1])
#构造一个标签平衡的训练集
length=15000
celebA_train_dataset=ConcatDataset([random_split(celebA_train_target1_dataset,[length,len(celebA_train_target1_dataset)-length])[0],random_split(celebA_train_target2_dataset,[length,len(celebA_train_target2_dataset)-length])[0]])
celebA_train_loader=DataLoader(dataset = celebA_train_dataset,
                              batch_size=batch_size,
                              shuffle=True)
print(len(celebA_train_target1_dataset))
print(len(celebA_train_target2_dataset))

21125
19567


In [13]:
#加载划分数据集
#男女验证集
indices=get_split_indices(celebA_val_dataset,group_index)
celebA_val_male1_dataset=Subset(celebA_val_dataset,indices[0])
celebA_val_male2_dataset=Subset(celebA_val_dataset,indices[1])
celebA_val_male1_loader=DataLoader(dataset = celebA_val_male1_dataset,
                              batch_size=batch_size,
                              shuffle=True)
celebA_val_male2_loader=DataLoader(dataset = celebA_val_male2_dataset,
                              batch_size=batch_size,
                              shuffle=True)
#男女性训练集
indices=get_split_indices(celebA_train_dataset,group_index)
celebA_train_male1_dataset=Subset(celebA_train_dataset,indices[0])
celebA_train_male2_dataset=Subset(celebA_train_dataset,indices[1])

In [14]:
#验证集
#性别1中biglips标签数据集
indices = get_split_indices(celebA_val_male1_dataset, target_index)
celebA_val_male1_biglips1_dataset = Subset(celebA_val_male1_dataset, indices[0])
celebA_val_male1_biglips2_dataset = Subset(celebA_val_male1_dataset, indices[1])
celebA_val_male1_biglips1_loader=DataLoader(dataset = celebA_val_male1_biglips1_dataset,
                              batch_size=batch_size,
                              shuffle=True)
celebA_val_male1_biglips2_loader=DataLoader(dataset = celebA_val_male1_biglips2_dataset,
                              batch_size=batch_size,
                              shuffle=True)
#性别2中biglips标签数据集
indices = get_split_indices(celebA_val_male2_dataset, target_index)
celebA_val_male2_biglips1_dataset = Subset(celebA_val_male2_dataset, indices[0])
celebA_val_male2_biglips2_dataset = Subset(celebA_val_male2_dataset, indices[1])
celebA_val_male2_biglips1_loader=DataLoader(dataset = celebA_val_male2_biglips1_dataset,
                              batch_size=batch_size,
                              shuffle=True)
celebA_val_male2_biglips2_loader=DataLoader(dataset = celebA_val_male2_biglips2_dataset,
                              batch_size=batch_size,
                              shuffle=True)
indices=get_split_indices(celebA_val_dataset,target_index)
celebA_val_biglips1_dataset=Subset(celebA_val_dataset,indices[0])
celebA_val_biglips2_dataset=Subset(celebA_val_dataset,indices[1])
celebA_val_biglips1_loader=DataLoader(dataset =celebA_val_biglips1_dataset,
                              batch_size=batch_size,
                              shuffle=True)
celebA_val_biglips2_loader=DataLoader(dataset =celebA_val_biglips2_dataset,
                              batch_size=batch_size,
                              shuffle=True)

In [15]:
#训练集
#性别1中biglips标签数据集
indices=get_split_indices(celebA_train_male1_dataset,target_index)
celebA_train_male1_biglips1_dataset=Subset(celebA_train_male1_dataset,indices[0])
celebA_train_male1_biglips2_dataset=Subset(celebA_train_male1_dataset,indices[1])
#性别2中biglips标签数据集
indices=get_split_indices(celebA_train_male2_dataset,target_index)
celebA_train_male2_biglips1_dataset=Subset(celebA_train_male2_dataset,indices[0])
celebA_train_male2_biglips2_dataset=Subset(celebA_train_male2_dataset,indices[1])

In [16]:
#shuffle数据集，含50个男正例、反例和女性正例反例的数据集，并打乱biglips
# celebA_train_balance_dataset=ConcatDataset([Subset(celebA_train_male1_biglips1_dataset,range(100)),
#                                             Subset(celebA_train_male1_biglips2_dataset,range(100)),
#                                             Subset(celebA_train_male2_biglips1_dataset,range(100)),
#                                             Subset(celebA_train_male2_biglips2_dataset,range(100))] )
# celebA_train_balance_dataset=ConcatDataset([Subset(celebA_train_male1_dataset,range(100)),
#                                             Subset(celebA_train_male2_dataset,range(100))] )
# celebA_train_balance_dataset=ConcatDataset([random_split(celebA_train_male1_dataset,[100,len(celebA_train_male1_dataset)-100])[0],
#                                             random_split(celebA_train_male2_dataset,[100,len(celebA_train_male2_dataset)-100])[0]] )
celebA_train_balance_dataset=ConcatDataset([random_split(celebA_train_male1_biglips1_dataset,[int(shuffle_len/4),len(celebA_train_male1_biglips1_dataset)-int(shuffle_len/4)])[0],
                                            random_split(celebA_train_male1_biglips2_dataset,[int(shuffle_len/4),len(celebA_train_male1_biglips2_dataset)-int(shuffle_len/4)])[0],
                                            random_split(celebA_train_male2_biglips1_dataset,[int(shuffle_len/4),len(celebA_train_male2_biglips1_dataset)-int(shuffle_len/4)])[0],
                                            random_split(celebA_train_male2_biglips2_dataset,[int(shuffle_len/4),len(celebA_train_male2_biglips2_dataset)-int(shuffle_len/4)])[0]] )
celebA_train_shuffle_biglips_dataset=shuffle_dataset(celebA_train_balance_dataset,target_index=target_index)
celebA_train_balance_dataloader=DataLoader(dataset = celebA_train_balance_dataset,
                              batch_size=batch_size,
                              shuffle=True)
celebA_train_shuffle_biglips_dataloader=DataLoader(dataset = celebA_train_shuffle_biglips_dataset,
                              batch_size=batch_size,
                              shuffle=True)

In [20]:
# 训练模型的方法定义
print('training on: ', device)
def test(loader, net,target_index):
    net.eval()
    acc = 0.0
    sum = 0.0
    loss_sum = 0
    for batch, (data, target) in enumerate(loader):
        data, target = data.to(device), target[:,target_index].to(device)
        output = net(data)
        loss = criterion(output, target)
        loss_sum += loss.item()
        _, predicted = output.max(1)
        sum += target.size(0)
        acc += predicted.eq(target).sum().item()
        # acc += torch.sum(torch.argmax(output, dim=1) == target).item()
        # sum += len(target)
        # loss_sum += loss.item()
    print('test  acc: %.2f%%, loss: %.4f' % (100 * acc / sum, loss_sum / (batch + 1)))
    return 100 * acc / sum, loss_sum / (batch + 1)

def train(loader, model, target_index, training_type):
    '''
    :param loader:
    :param model:
    :param target_index: 标签下标
    :param training_type: 模型名称
    :return:
    '''
    model.train()
    acc = 0.0
    sum = 0.0
    loss_sum = 0

    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9, weight_decay=5e-4)
    for batch, (data, target) in tqdm.tqdm( enumerate(loader),desc="模型训练中：", total=len(loader)):
        data, target = data.to(device), target[:,target_index].type(torch.LongTensor).to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()
        _, predicted = output.max(1)
        sum += target.size(0)
        acc += predicted.eq(target).sum().item()

        # acc += torch.sum(torch.argmax(output, dim=1) == target).item()
        # sum += len(target)
        # loss_sum += loss.item()

        # if batch % 200 == 0:
        #     print('\tbatch: %d, loss: %.4f' % (batch, loss.item()))
    print('train acc: %.2f%%, loss: %.4f' % (100 * acc / sum, loss_sum / (batch + 1)))
    torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': loss,
            }, "../models/" + str(training_type) + "_checkpoint.pth")

def load_model(model_path):
    #加载模型
    net = VGG('VGG16').to(device)
    checkpoint = torch.load(model_path)
    net.load_state_dict(checkpoint['model_state_dict'])
    return net
def CelebA_test(model,target_index):
    # print("全部测试集：")
    # test(celebA_val_loader,model,target_index=target_index)
    # print("性别1测试集：")
    # test(celebA_val_male1_loader,model,target_index=target_index)
    # print("性别2测试集：")
    # test(celebA_val_male2_loader,model,target_index=target_index)
    print("标签1测试集：")
    test(celebA_val_biglips1_loader,model,target_index=target_index)
    print("标签2测试集：")
    test(celebA_val_biglips2_loader,model,target_index=target_index)

training on:  cuda


In [ ]:
%%time
#原始训练
for epoch in range(200):
        print('epoch: %d' % epoch)
        train(celebA_train_loader,net,target_index=target_index,training_type="celebA_origin_smiling")
        CelebA_test(net,target_index=target_index)

epoch: 0


模型训练中：: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 235/235 [02:14<00:00,  1.74it/s]


train acc: 51.22%, loss: 1.2805
标签1测试集：
test  acc: 79.68%, loss: 0.6420
标签2测试集：
test  acc: 28.20%, loss: 0.8233
epoch: 1


模型训练中：: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 235/235 [02:15<00:00,  1.73it/s]


train acc: 54.00%, loss: 0.7375
标签1测试集：
test  acc: 68.05%, loss: 0.6285
标签2测试集：
test  acc: 45.12%, loss: 0.7373
epoch: 2


模型训练中：: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 235/235 [02:17<00:00,  1.71it/s]


train acc: 56.03%, loss: 0.6847
标签1测试集：
test  acc: 97.16%, loss: 0.5084
标签2测试集：
test  acc: 4.70%, loss: 0.9070
epoch: 3


模型训练中：: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 235/235 [02:16<00:00,  1.72it/s]


train acc: 56.27%, loss: 0.6847
标签1测试集：


In [ ]:
net=load_model('models/celebA_origin_smiling_checkpoint.pth')

In [ ]:
CelebA_test(net,target_index=target_index)

In [ ]:
#混淆训练,celebA数据集target为biglips
for epoch in range(10):
        print('epoch: %d' % epoch)
        train(celebA_train_shuffle_biglips_dataloader,net,target_index=target_index,training_type="celebA_shuffle_bignose")

In [ ]:
net=load_model('models/celebA_shuffle_crop_biglips_checkpoint.pth')

In [ ]:
CelebA_test(net,target_index=target_index)

In [ ]:
#恢复训练,celebA数据集target为biglips
for epoch in range(10):
        print('epoch: %d' % epoch)
        train(celebA_train_balance_dataloader,net,target_index=target_index,training_type="celebA_balance_bignose")

In [ ]:
net=load_model('models/celebA_balance_biglips_checkpoint.pth')

In [ ]:
CelebA_test(net,target_index=target_index)